# 06 · Monitoring and comparing the strategies

Read the governance signals back and turn them into alerts, then run every strategy side by side.

| | |
| --- | --- |
| **Needs** | `00_setup.ipynb` in the same folder. The first code cell runs it for you. |
| **LLM** | Not used in this notebook. |
| **Previous / next** | [`05_governed_retrieval.ipynb`](05_governed_retrieval.ipynb) / none, this is the last one |

*The code mirrors the [`rag-retrieval` journey](https://github.com/instaclustr/labs/tree/main/rag-retrieval) in `instaclustr/labs`. Lines marked "Mirrors" name the file in the journey that each piece comes from. This notebook only searches: it never writes to the cluster.*

### Run the setup

The next cell runs `00_setup.ipynb`. It connects to the cluster, loads the two small models, and defines the helper functions used below. You should see `[PASS]` lines. If you see `[FAIL]`, fix that before going on, using the notes in `00_setup.ipynb`.

In [ ]:
%run ./00_setup.ipynb

### Recap from earlier notebooks

This notebook uses the retrieval functions from notebooks 01 to 05 (vector, BM25, hybrid, grounded refinement and governed retrieval) and the governance thresholds. They are repeated in the next cell, unchanged, so this notebook runs on its own. The new material is below it.

In [ ]:
def vector_search(question, top_k=5, candidates=25, index=None):
    body = {"size": top_k, "_source": RECIPE_FIELDS,
            "query": {"knn": {"embedding": {"vector": embed(question), "k": candidates}}}}
    return normalize_hits(search(index or IDX["vector_baseline"], body))


def bm25_query(text, k, entities=None, tier=None, exclude=None, require=None, recipe_ids=None):
    """BM25 over name and text (`source` and `entities` are exact-match fields), with optional hard filters and an entity boost."""
    filters, must_not, should = [], [], []
    if recipe_ids:
        filters.append({"terms": {"recipe_id": list(recipe_ids)}})
    if tier:
        filters.append({"term": {"tier": tier}})
    for field, values in (exclude or {}).items():
        must_not.append({"terms": {field: [v.lower() for v in values]}})
    for field, values in (require or {}).items():
        filters += [{"term": {field: v.lower()}} for v in values]
    if entities:
        should.append({"constant_score": {"filter": {"terms": {"entities": entities}}, "boost": 6.0}})
    must = [{"multi_match": {"query": text, "fields": ["recipe_name^4", "entities^3", "text^2", "source"],
                             "type": "best_fields", "operator": "or"}}]
    bool_query = {"must": must, "filter": filters, "must_not": must_not}
    if should:
        bool_query["should"] = should
    return {"size": k, "_source": RECIPE_FIELDS + ["entities"], "query": {"bool": bool_query}}


def bm25_search(question, k=10, index=None, **kw):
    entities = extract_entities(question)
    return normalize_hits(search(index or IDX["bm25_baseline"], bm25_query(question, k, entities=entities, **kw))), entities


def hybrid_search(question, k=5):
    body = {"size": k, "_source": RECIPE_FIELDS,
            "query": {"hybrid": {"queries": [
                {"match": {"text": question}},
                {"knn": {"embedding": {"vector": embed(question), "k": k}}},
            ]}}}
    return normalize_hits(search(IDX["hybrid"], body, search_pipeline="hybrid-search-pipeline"))


def vector_refine(question, recipe_ids, top_k=5, exclude=None, pool=300):
    """kNN restricted to the grounding candidates. `pool` must cover the whole corpus, because the ID filter
    is applied after the nearest-neighbour search."""
    filters = [{"terms": {"recipe_id": list(recipe_ids)}}]
    must_not = [{"terms": {f: [v.lower() for v in vs]}} for f, vs in (exclude or {}).items()]
    body = {"size": top_k, "_source": RECIPE_FIELDS,
            "query": {"bool": {"must": [{"knn": {"embedding": {"vector": embed(question), "k": max(pool, len(recipe_ids))}}}],
                               "filter": filters, "must_not": must_not}}}
    return normalize_hits(search(IDX["vector"], body))


def ground_and_refine(question, top_k=5, candidate_k=25, exclude=None, require=None):
    entities = extract_entities(question)
    grounding = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities,
                                                              exclude=exclude, require=require)))
    ids = [h["recipe_id"] for h in grounding if h.get("recipe_id")]
    refined = vector_refine(question, ids, top_k, exclude) if ids else []
    return grounding, refined


MIN_HOT_SCORE, MIN_HOT_CANDIDATES, REFINE_ANN_POOL = 15.0, 3, 300


CITATION_RE = re.compile(r"\[(R\d+)\]")


DOMAIN_RE = re.compile(r"\b(?=[\w-]*[A-Za-z])[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}\b")


def governed_retrieve(question, top_k=5, candidate_k=25, exclude=None, require=None):
    entities = extract_entities(question)
    hot = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities, tier="hot",
                                                        exclude=exclude, require=require)))
    top_score = hot[0]["score"] if hot else 0.0
    named = [m.group(0).lower() for m in DOMAIN_RE.finditer(question)]
    hot_sources = {(h.get("source") or "").lower() for h in hot}
    named_missing = bool(named) and not any(tok in src for tok in named for src in hot_sources)

    escalate = (not hot or len(hot) < MIN_HOT_CANDIDATES or top_score < MIN_HOT_SCORE or named_missing)
    reason = None
    if escalate:
        reason = ("no_hot_candidates" if not hot else "named_source_not_in_hot_tier" if named_missing
                  else "low_confidence_hot_score")
        grounding = normalize_hits(search(IDX["bm25"], bm25_query(question, candidate_k, entities=entities,
                                                                  exclude=exclude, require=require)))
        tiers = ["hot", "lt"]
    else:
        grounding, tiers = hot, ["hot"]

    ids = [h["recipe_id"] for h in grounding if h.get("recipe_id")]
    refined = vector_refine(question, ids, top_k, exclude, pool=REFINE_ANN_POOL) if ids else []
    refined = [{**h, "handle": f"R{i}"} for i, h in enumerate(refined, start=1)]
    meta = {"tiers_searched": tiers, "escalated": escalate, "escalation_reason": reason,
            "hot_hit_count": len(hot), "hot_top_score": round(top_score, 3),
            "evidence_handles": [h["handle"] for h in refined]}
    return grounding, refined, meta, entities

## Operations

Governance only helps if someone reads the audit trail. This step reads it back and turns it into **alerts**, which is what the journey's `monitor.py` does on a schedule.

An alert fires if:

- any answer contained a **hallucinated citation**,
- the **escalation rate** in the lookback window is above a threshold (0.5 by default), which can signal tiering drift, such as newly loaded "hot" data missing a caution it should have,
- the entity-tagging service is **not responding**. In this notebook spaCy runs in-process, so that check is simply "did the model load".

*Mirrors:* `scripts/operationalize-production/src/monitor.py` and `audit_log_governance_stats()`.

In [ ]:
def audit_stats(index, lookback_hours=24):
    body = {"size": 0, "query": {"range": {"timestamp": {"gte": f"now-{int(lookback_hours)}h"}}},
            "aggs": {"escalated": {"filters": {"filters": {"true": {"term": {"escalated": True}},
                                                           "false": {"term": {"escalated": False}}}}},
                     "reasons": {"terms": {"field": "escalation_reason", "size": 20, "missing": "none"}},
                     "citations_invalid": {"filter": {"term": {"citations_valid": False}}}}}
    r = search(index, body)
    total = r["hits"]["total"]["value"]
    if not total:
        return {"total": 0, "escalated": 0, "escalation_rate": 0.0, "escalation_reasons": {},
                "citations_invalid": 0, "hallucination_rate": 0.0}
    a = r["aggregations"]
    esc = a["escalated"]["buckets"]["true"]["doc_count"]
    bad = a["citations_invalid"]["doc_count"]
    return {"total": total, "escalated": esc, "escalation_rate": esc / total,
            "escalation_reasons": {b["key"]: b["doc_count"] for b in a["reasons"]["buckets"] if b["key"] != "none"},
            "citations_invalid": bad, "hallucination_rate": bad / total}


def evaluate_alerts(stats, ner_healthy, max_escalation_rate=0.5):
    alerts = []
    if stats["total"]:
        if stats["citations_invalid"] > 0:
            alerts.append(f"{stats['citations_invalid']} of {stats['total']} answer(s) contained a hallucinated citation")
        if stats["escalation_rate"] > max_escalation_rate:
            alerts.append(f"escalation rate {stats['escalation_rate']:.1%} exceeds {max_escalation_rate:.1%} "
                          "-- possible hot/LT tiering drift")
    if not ner_healthy:
        alerts.append("entity tagging is not available -- entity boosting is silently disabled")
    return alerts


LOOKBACK_HOURS = 24 * 30   # the shared audit log is sparse; look back a month so there is something to read
stats = audit_stats(IDX["audit"], LOOKBACK_HOURS)
print(f"audit records in the last {LOOKBACK_HOURS}h: {stats['total']}")
print(f"  escalation rate:     {stats['escalation_rate']:.1%}  reasons={stats['escalation_reasons']}")
print(f"  hallucination rate:  {stats['hallucination_rate']:.1%}")
alerts = evaluate_alerts(stats, ner_healthy=nlp is not None)
print("ALERTS:" if alerts else "No governance alerts.")
for a in alerts:
    print("  -", a)

**Expected:** a count of recent audit records (the shared log is small), an escalation rate with reasons, and a hallucination rate. With only a handful of records, one escalation is a huge percentage, so the escalation alert can fire here simply because the sample is tiny. Whether an alert fires depends on what earlier users did. The point is the loop: *write a record, read the records, alert on the signal.*

### The two controls that go with it

**Retention.** Audit records grow without bound. The journey attaches an index-lifecycle (ISM) policy named `recipes-audit-log-retention` that **deletes the whole audit index once it is 180 days old**. The cell below checks whether the cluster has that policy and whether it is attached to the audit index. It only reads.

**Policy change control.** The tier thresholds (`15.0` score, `3` candidates) are governance decisions. The journey records them with an approver and date in `policy_history.json`, and a drift check compares the running values to the last approved version.

In [ ]:
POLICY_ID = "recipes-audit-log-retention"
try:
    listing = client.transport.perform_request("GET", "/_plugins/_ism/policies")
    policy_ids = [p["_id"] for p in listing.get("policies", [])]
    explain = client.transport.perform_request("GET", f"/_plugins/_ism/explain/{IDX['audit']}")
    attached = explain.get(IDX["audit"], {}).get("index.plugins.index_state_management.policy_id")
    print(f"ISM policies on this cluster: {policy_ids or 'none'}")
    if POLICY_ID in policy_ids:
        states = client.transport.perform_request("GET", f"/_plugins/_ism/policies/{POLICY_ID}")["policy"]["states"]
        for s in states:
            for t in s.get("transitions", []):
                print(f"  {POLICY_ID}: {s['name']} -> {t['state_name']} when {t['conditions']}")
    print(f"policy attached to the audit index '{IDX['audit']}': {attached or 'no'}")
    if attached != POLICY_ID:
        print("[WARN] no retention policy is attached to this audit index, so it is not bounded.")
except Exception as exc:  # noqa: BLE001
    print(f"(could not read the ISM state with this account: {type(exc).__name__})")

APPROVED = {"version": 1, "tier_min_hot_score": 15.0, "tier_min_hot_candidates": 3}
running = {"tier_min_hot_score": MIN_HOT_SCORE, "tier_min_hot_candidates": MIN_HOT_CANDIDATES}
drift = {k: (APPROVED[k], v) for k, v in running.items() if APPROVED[k] != v}
print("[OK] running thresholds match the approved policy version", APPROVED["version"] if not drift
      else f"[DRIFT] {drift}")

**Expected:** on a cluster where the journey's Task 7 has been run, you see the `recipes-audit-log-retention` policy and its `delete` transition after 180 days. On this shared cluster that policy has **not** been created yet, so you will see `none` and a `[WARN]`: the shared audit log is unbounded. That is a real finding, and it is the kind of gap the journey's Task 7 closes. Creating the policy needs write access, so this notebook only reports it. Then you see `[OK]`, because the thresholds in this notebook match the approved version.

**Takeaway.** Retrieval quality is only half of "a RAG system you can defend". The other half is being able to say what was searched, why, and who approved the rules, and to notice when it drifts.

## Compare the strategies

Now the same two questions go through every retrieval strategy, side by side. The table asks one thing: **did the recipe the question names show up in the results?** No LLM is needed.

Then a second table runs the exclusion request from notebook 04 through the hybrid query and the grounded pipeline, to show what each can *guarantee*.

In [ ]:
strategies = {
    "01 Vector": lambda q: vector_search(q, 5),
    "02 BM25 + NER": lambda q: bm25_search(q, 10)[0],
    "03 Hybrid (native)": lambda q: hybrid_search(q, 5),
    "04 NER + rerank": lambda q: ground_and_refine(q, 5)[1],
    "05 Governed": lambda q: governed_retrieve(q, 5)[1],
}
rows = []
for name, retrieve in strategies.items():
    g = retrieve(Q_GRATEFUL)
    w = retrieve(Q_WOMENS)
    rows.append({"strategy": name,
                 "thegratefulgirlcooks found": has_source(g, "thegratefulgirlcooks"),
                 "womensweeklyfood found": has_source(w, "womensweeklyfood")})
pd.DataFrame(rows).set_index("strategy")

**Expected:** Notebook 01's vector search misses the `thegratefulgirlcooks.com` recipe for its own question. Every later strategy finds both. Scores and borderline ranks can vary a little between runs.

### What each strategy can guarantee

"No sulfites" is a **hard** requirement. This table counts retrieved recipes that carry a Sulfites caution when the request excludes them.

In [ ]:
def count_sulfites(hits):
    return sum("sulfites" in [c.lower() for c in cautions_of(h)] for h in hits)


Q = "Suggest a British fish and chips recipe with no sulfites."
excl = {"cautions": ["sulfites"]}
vec_hits = vector_search(Q, 5)
hyb_hits = hybrid_search(Q, 5)
rows = [
    {"strategy": "01 Vector (no filter possible)", "retrieved": len(vec_hits), "with Sulfites": count_sulfites(vec_hits)},
    {"strategy": "03 Hybrid (native), scores only", "retrieved": len(hyb_hits), "with Sulfites": count_sulfites(hyb_hits)},
]
g, r = ground_and_refine(Q, 5, exclude=excl)
rows.append({"strategy": "04 NER + rerank, hard exclusion", "retrieved": len(r), "with Sulfites": count_sulfites(r)})
pd.DataFrame(rows).set_index("strategy")

**Expected:** the vector and hybrid rows can still contain recipes that carry a Sulfites caution, because a similarity score is not a rule. The hard-exclusion row has **0**, and always will, because the search itself rejects them.

## Where to go next

- The full journey, with the ingest scripts, the audit logging, and the retention policy, is in [`rag-retrieval` in instaclustr/labs](https://github.com/instaclustr/labs/tree/main/rag-retrieval). It writes indexes and needs read-write access to a cluster.
- To run these notebooks on your own cluster, set `USE_SHARED_READONLY_CLUSTER = False` in `00_setup.ipynb`, put your `OPENSEARCH_*` settings in `.env`, and load the journey's data with its ingest steps first.
- Ideas to explore: change `top_k`, add `require` filters, try questions the journey never asked, and watch which strategies break.

### Checkpoint

You should be able to say what each strategy can and cannot guarantee, and what the audit log needs around it (retention and a change record for the thresholds).

That is the last notebook.